In [681]:
from dash import Dash, dcc, html, Input, Output, State
import dash_bootstrap_components as dbc
from dash import dcc, html, Input, Output, State
import dash_bootstrap_components as dbc
import plotly.graph_objects as go
import numpy as np
import traceback

In [682]:
#pip install reportlab


In [683]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
import datetime


In [684]:
from pathlib import Path
import numpy as np
import wfdb # type: ignore
import matplotlib.pyplot as plt
import pandas as pd
import csv
import pywt # type: ignore
from scipy.signal import hilbert
from scipy.signal import find_peaks
import ast
from scipy.stats import skew, kurtosis

In [685]:
df_pacientes=pd.read_csv("C:\\Users\\Ana\\Documents\\4º GITT+BA (2025-2026)\\TFG\\TFG\\ptbxl_database.csv")
df_scp=pd.read_csv("C:\\Users\\Ana\\Documents\\4º GITT+BA (2025-2026)\\TFG\\TFG\\scp_statements.csv",sep=";")

In [686]:
#variables fijas
SECONDS_TO_SHOW = 10
fs=500
pre=int(0.2*fs)#100 muestras antes del pico R
post=int(0.4*fs)#200 muestras después del pico R
wavelet="db6"
nivel=8
MAX_LATIDOS = 10#Lo reducimos a 10 porque así hay más datos
LONGITUD_LATIDO = 300

In [687]:
#Funciones 
def strip_wfdb_suffix(p: Path) -> Path:
    p = str(p).strip().strip('"').strip("'")
    
    p=Path(p)
    while p.suffix in (".dat", ".hea"):
        p = p.with_suffix("")
    return p

def leer_ecg(BASE):
    BASE = strip_wfdb_suffix(BASE)
    p_signal, fields = wfdb.rdsamp(str(BASE))
    fs = float(fields["fs"])
    lead_names = fields["sig_name"]
    n, L = p_signal.shape
    n_show = min(n, int(SECONDS_TO_SHOW * fs))
    x = np.arange(n_show) / fs  # eje tiempo en segundos
    sig = p_signal[:n_show, :]

    # Calcula un espaciado vertical cómodo (basado en mediana del pico-a-pico por derivación)
    ptp_per_lead = np.ptp(sig, axis=0)  # pico-a-pico por derivación
    spacing = 1.2 * np.median(ptp_per_lead) if np.median(ptp_per_lead) > 0 else 100.0

    return L,x,sig,spacing,BASE,n_show,lead_names,fs

def graficar_ecg(L,x,sig,spacing,BASE,n_show,lead_names,fs):#Dash no renderiza plt hay que usar plotly
    fig=go.Figure()
    for i in range(sig.shape[1]):  # número real de derivaciones

        fig.add_trace(
            go.Scatter(
                x=x,
                y=sig[:, i] + i * spacing,
                mode='lines',
                name=lead_names[i],
                line=dict(width=1)
            )
        )

    fig.update_layout(
        title=f"ECG 12 derivaciones — {Path(BASE).name} — Fs={fs:.0f} Hz",

        height=900,

        plot_bgcolor='white',
        paper_bgcolor='white',

        xaxis_title="Tiempo (s)",
        yaxis_title="Derivaciones",

        showlegend=False,

        margin=dict(l=60, r=30, t=60, b=40)
    )

    fig.update_yaxes(
        tickvals=[i * spacing for i in range(sig.shape[1])],
        ticktext=lead_names
    )

    return fig

def detectar_picos_R(derivacion_I, wavelet, nivel):
    coeffs = pywt.wavedec(derivacion_I, wavelet, level=nivel)
    coeffs_qrs = [np.zeros_like(c) for c in coeffs]
    coeffs_qrs[4] = coeffs[4]  # d5
    coeffs_qrs[5] = coeffs[5]  # d4
    qrs_signal = pywt.waverec(coeffs_qrs, 'db6')
    qrs_signal = qrs_signal[:len(derivacion_I)]
    analytic_signal = hilbert(qrs_signal)
    envelope = np.abs(analytic_signal)
    umbral = np.mean(envelope) + 1.0 * np.std(envelope)
    peaks, _ = find_peaks(envelope, height=umbral, distance=200)
    return peaks

def extraer_latidos(paciente,derivacion,pre,post):
    latidos_recortados=[]
    for latidos in paciente:
        if latidos-pre>=0 and latidos+post<len(derivacion):
            latido=derivacion[latidos-pre:latidos+post]
            latidos_recortados.append(latido)
    latidos_recortados=np.array(latidos_recortados)
    return latidos_recortados

In [688]:
def extraer_features(latidos, picos_R, fs=500):
    features = []

    # ── 1. TEMPORALES / HRV ───────────────────────────────────────────────────
    intervalos_RR = np.diff(picos_R) / fs
    features.append(np.mean(intervalos_RR))
    features.append(np.std(intervalos_RR))
    features.append(np.min(intervalos_RR))
    features.append(np.max(intervalos_RR))
    features.append(np.max(intervalos_RR) - np.min(intervalos_RR))
    features.append(60 / np.mean(intervalos_RR))
    rmssd = np.sqrt(np.mean(np.diff(intervalos_RR) ** 2)) if len(intervalos_RR) > 1 else 0.0
    features.append(rmssd)

    # ── 2. MORFOLÓGICAS ───────────────────────────────────────────────────────
    amp_R, amp_T, area_QRS, dur_QRS, amp_Q, st_nivel = [], [], [], [], [], []
    for latido in latidos:
        pico_R_idx = 100
        amp_R.append(latido[pico_R_idx])
        amp_T.append(np.max(latido[150:250]))
        zona_QRS = latido[75:125]
        area_QRS.append(np.trapz(np.abs(zona_QRS)))
        umbral_QRS = 0.5 * latido[pico_R_idx]
        dur_QRS.append(np.sum(zona_QRS > umbral_QRS))
        amp_Q.append(np.min(latido[75:100]))
        st_nivel.append(np.mean(latido[125:155]))

    for stat_list in [amp_R, amp_T, area_QRS, dur_QRS, amp_Q, st_nivel]:
        features.append(np.mean(stat_list))
        features.append(np.std(stat_list))

    # ── 3. ESTADÍSTICAS ───────────────────────────────────────────────────────
    for fn in [np.mean, np.std, skew, kurtosis, np.max, np.min]:
        vals = [fn(l) for l in latidos]
        features.append(np.mean(vals))
        features.append(np.std(vals))

    # ── 4. FRECUENCIALES ──────────────────────────────────────────────────────
    energias_d4, energias_d5, energias_d6 = [], [], []
    for latido in latidos:
        coeffs = pywt.wavedec(latido, 'db6', level=8)
        energias_d4.append(np.sum(coeffs[5] ** 2))
        energias_d5.append(np.sum(coeffs[4] ** 2))
        energias_d6.append(np.sum(coeffs[3] ** 2))

    for energias in [energias_d4, energias_d5, energias_d6]:
        features.append(np.mean(energias))
        features.append(np.std(energias))

    ratio_LF_HF = np.mean(energias_d6) / (np.mean(energias_d4) + 1e-8)
    features.append(ratio_LF_HF)

    return np.array(features)

def calcular_eje_electrico(latidos_12):
    qrs_I   = latidos_12[0]
    qrs_aVF = latidos_12[5]

    amp_I_lista   = []
    amp_aVF_lista = []

    for latido_I, latido_aVF in zip(qrs_I, qrs_aVF):
        zona_I   = latido_I[75:125]
        zona_aVF = latido_aVF[75:125]
        amp_I_lista.append(np.max(zona_I) - np.abs(np.min(zona_I)))
        amp_aVF_lista.append(np.max(zona_aVF) - np.abs(np.min(zona_aVF)))

    amp_I   = np.mean(amp_I_lista)
    amp_aVF = np.mean(amp_aVF_lista)        
    return np.degrees(np.arctan2(amp_aVF, amp_I))

In [689]:
def construir_matriz_paciente(latidos,max_latidos,longitud_latido):
    matriz_paciente = np.zeros((max_latidos, longitud_latido))
    for i in range(min(len(latidos), max_latidos)):
        latido = latidos[i]
        
        # por seguridad si alguna ventana sale rara
        if len(latido) >= longitud_latido:
            matriz_paciente[i] = latido[:longitud_latido]
        else:
            matriz_paciente[i, :len(latido)] = latido
    return matriz_paciente

In [690]:
def preparar_resumen(BASE):

    resumen={}

    # ─────────────────────────────
    # 1. LEER ECG
    # ─────────────────────────────
    L, x, sig, spacing, BASE, n_show, lead_names, fs = leer_ecg(BASE)
    p_signal_t = sig.T  # (12, N)

    # ─────────────────────────────
    # 2. R PEAKS
    # ─────────────────────────────
    ref = p_signal_t[0]
    Rloc = detectar_picos_R(ref, wavelet, nivel)

    # ─────────────────────────────
    # 3. HRV (HR, RR, SDNN)
    # ─────────────────────────────
    if len(Rloc) < 2:
        hr = 0
        rr_mean = 0
        sdnn = 0
    else:
        rr = np.diff(Rloc) / fs
        rr_mean = np.mean(rr)
        sdnn = np.std(rr)
        hr = 60 / rr_mean

    # ─────────────────────────────
    # 4. QRS duración (lead I)
    # ─────────────────────────────
    qrs_vals = []
    for r in Rloc:
        if r - 50 >= 0 and r + 50 < len(ref):
            ventana = ref[r-50:r+50]
            umbral = 0.5 * np.max(ventana)
            qrs_vals.append(np.sum(ventana > umbral))

    qrs_mean = np.mean(qrs_vals) if len(qrs_vals) > 0 else 0

    # ─────────────────────────────
    # 5. ST segment (lead I)
    # ─────────────────────────────
    st_vals = []
    lead = p_signal_t[0]

    for r in Rloc:
        if r + 80 < len(lead):
            st_vals.append(np.mean(lead[r+60:r+80]))

    st_mean = np.mean(st_vals) if len(st_vals) > 0 else 0

    # ─────────────────────────────
    # 6. R amplitude (lead I)
    # ─────────────────────────────
    r_amp_vals = []
    for r in Rloc:
        if r < len(ref):
            r_amp_vals.append(ref[r])

    r_mean = np.mean(r_amp_vals) if len(r_amp_vals) > 0 else 0

    # ─────────────────────────────
    # 7. EJE ELÉCTRICO (12 derivaciones)
    # ─────────────────────────────
    latidos_12 = []

    for d in range(12):
        deriv = p_signal_t[d]

        latidos_d = extraer_latidos(Rloc, deriv, pre, post)
        latidos_d = latidos_d[:MAX_LATIDOS]

        matriz_d = construir_matriz_paciente(
            latidos_d,
            MAX_LATIDOS,
            LONGITUD_LATIDO
        )

        latidos_12.append(matriz_d)

    latidos_12 = np.stack(latidos_12, axis=0)

    eje = calcular_eje_electrico(latidos_12)

    name = Path(BASE).stem  # sin extensión
    mask = df_pacientes['filename_hr'].str.contains(name, case=False, na=False, regex=False)
    fila = df_pacientes[mask]

    edad = None
    sexo = None

    if len(fila) > 0:
        edad = fila['age'].values[0]
        sexo = 1.0 if fila['sex'].values[0] == 1 else 0.0 

    resumen = {
        "hr": hr,
        "rr_mean": rr_mean,
        "sdnn": sdnn,
        "qrs": qrs_mean,
        "st": st_mean,
        "r_amp": r_mean,
        "eje": eje,
        "sexo":sexo,
        "edad":edad
    }


    # ─────────────────────────────
    # 8. OUTPUT CLÍNICO FINAL
    # ─────────────────────────────
    return resumen,len(Rloc)


In [691]:
def generar_resumen_dashboard(resumen, n_latidos):

    return dbc.Card([
        dbc.CardBody([

            # ─────────────────────────────
            # SEXO + EDAD + LATIDOS
            # ─────────────────────────────
            
            dbc.Row([
                dbc.Col(
                    dbc.Card([
                        dbc.CardBody([
                            html.H3(f"{n_latidos}"),
                            html.P("Latidos detectados")
                        ])
                    ], color='light'),
                    width=2
                ),
                dbc.Col(
                    dbc.Card([
                        dbc.CardBody([
                            html.H3(f"{resumen["edad"]}"),
                            html.P("edad")
                        ])
                    ], color='light'),
                    width=2
                ),
                dbc.Col(
                    dbc.Card([
                        dbc.CardBody([
                            html.H3(f"{resumen["sexo"]}"),
                            html.P("sexo")
                        ])
                    ], color='light'),
                    width=2
                ),

            ]),

            html.Br(),

            # ─────────────────────────────
            # VITALES + LATIDOS
            # ─────────────────────────────
            
            dbc.Row([

                dbc.Col(
                    dbc.Card([
                        dbc.CardBody([
                            html.H3(f"{round(resumen['hr'],1)}"),
                            html.P("HR (bpm)")
                        ])
                    ], color='light'),
                    width=2
                ),

                dbc.Col(
                    dbc.Card([
                        dbc.CardBody([
                            html.H3(f"{round(resumen['rr_mean'],3)}"),
                            html.P("RR medio (s)")
                        ])
                    ], color='light'),
                    width=2
                ),

                dbc.Col(
                    dbc.Card([
                        dbc.CardBody([
                            html.H3(f"{round(resumen['sdnn'],3)}"),
                            html.P("HRV (SDNN)")
                        ])
                    ], color='light'),
                    width=2
                ),

                dbc.Col(
                    dbc.Card([
                        dbc.CardBody([
                            html.H3(f"{round(resumen['eje'],1)}°"),
                            html.P("Eje eléctrico")
                        ])
                    ], color='light'),
                    width=2
                ),

            ]),

            html.Br(),

            # ─────────────────────────────
            # MORFOLOGÍA
            # ─────────────────────────────
            dbc.Row([

                dbc.Col(
                    dbc.Card([
                        dbc.CardBody([
                            html.H3(f"{round(resumen['qrs'],1)}"),
                            html.P("QRS")
                        ])
                    ], color='light'),
                    width=4
                ),

                dbc.Col(
                    dbc.Card([
                        dbc.CardBody([
                            html.H3(f"{round(resumen['st'],3)}"),
                            html.P("ST segment")
                        ])
                    ], color='light'),
                    width=4
                ),

                dbc.Col(
                    dbc.Card([
                        dbc.CardBody([
                            html.H3(f"{round(resumen['r_amp'],2)}"),
                            html.P("R amplitude")
                        ])
                    ], color='light'),
                    width=4
                ),

            ])

        ])
    ], style={'marginBottom': '20px'})

In [692]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def pintar_latidos_12(latidos_12, Rloc, pre, post, lead_names):

    """
    latidos_12: (12, n_latidos, longitud_latido)
    Rloc: picos R (solo usado si quieres marcar línea en 0)
    """

    n_leads = 12

    fig = make_subplots(
        rows=4, cols=3,
        subplot_titles=lead_names[:12],
        shared_xaxes=True,
        shared_yaxes=False
    )

    # eje temporal relativo al pico R
    tiempo = np.linspace(-pre/fs, post/fs, latidos_12.shape[2])

    row, col = 1, 1

    for d in range(n_leads):

        latidos = latidos_12[d]  # (n_latidos, samples)

        if len(latidos) == 0:
            row, col = _next_cell(row, col)
            continue

        # ── todos los latidos ──
        for beat in latidos:
            fig.add_trace(
                go.Scatter(
                    x=tiempo,
                    y=beat,
                    mode='lines',
                    line=dict(width=1),
                    opacity=0.3,
                    showlegend=False
                ),
                row=row,
                col=col
            )

        # ── latido medio ──
        mean_beat = np.mean(latidos, axis=0)

        fig.add_trace(
            go.Scatter(
                x=tiempo,
                y=mean_beat,
                mode='lines',
                line=dict(width=2, color='red'),
                name='Latido medio' if d == 0 else None,
                showlegend=(d == 0)
            ),
            row=row,
            col=col
        )

        # siguiente subplot
        row, col = _next_cell(row, col)

    fig.update_layout(
        title="Latidos alineados por derivación (12 leads)",
        height=900,
        plot_bgcolor="white",
        paper_bgcolor="white",
        showlegend=True
    )

    return fig


def _next_cell(row, col):
    col += 1
    if col > 3:
        col = 1
        row += 1
    return row, col

In [693]:
#Cargamos el modelo de mlp
import torch
import json
import numpy as np

# ── cargar info del modelo ──
with open("mlp_info.json", "r") as f:
    info = json.load(f)

ENFERMEDADES_71 = np.load("ENFERMEDADES_71.npy", allow_pickle=True).tolist()
UMBRAL_71 = np.load("UMBRAL_71.npy", allow_pickle=True).item()

In [694]:
print(info["input_size"])

459


In [695]:
import torch.nn as nn
class MLPRegressor(nn.Module):
        def __init__(self, input_size, output_size):
            super().__init__()
            self.red = nn.Sequential(
                nn.Linear(input_size, 256),
                nn.BatchNorm1d(256),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(256, 128),
                nn.BatchNorm1d(128),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(64, output_size),
                #nn.Sigmoid()
            )
        def forward(self, x):
            return self.red(x)

In [696]:
model = MLPRegressor(
    input_size=info["input_size"],
    output_size=info["output_size"]
)

model.load_state_dict(torch.load("mlp_ecg.pth", map_location="cpu"))
model.eval()

MLPRegressor(
  (red): Sequential(
    (0): Linear(in_features=459, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): ReLU()
    (10): Dropout(p=0.2, inplace=False)
    (11): Linear(in_features=64, out_features=71, bias=True)
  )
)

In [697]:
def predecir_desde_ecg(BASE,resumen):

    L, x, sig, spacing, BASE, n_show, lead_names, fs = leer_ecg(BASE)
    p_signal_t = sig.T

    ref = p_signal_t[0]
    Rloc = detectar_picos_R(ref, wavelet, nivel)

    latidos_12 = []

    features_todas = []

    for d in range(12):
        deriv = p_signal_t[d]

        latidos_d = extraer_latidos(Rloc, deriv, pre, post)
        latidos_d = latidos_d[:MAX_LATIDOS]

        latidos_12.append(
            construir_matriz_paciente(latidos_d, MAX_LATIDOS, LONGITUD_LATIDO)
        )

        features_d = extraer_features(latidos_d, Rloc, fs)
        features_todas.append(features_d)

    latidos_12 = np.stack(latidos_12, axis=0)

    features_concat = np.concatenate(features_todas)


    # si tienes edad/sexo disponibles en dashboard, pásalos aquí
    edad = resumen["edad"]
    sexo = resumen["sexo"]
    eje=resumen["eje"]

    X = np.append(features_concat, [eje, edad, sexo]).reshape(1, -1)

    print("Shape input:", X.shape)
    print("Esperado:", info["input_size"])

    with torch.no_grad():
        x_t = torch.tensor(X, dtype=torch.float32)
        pred = torch.sigmoid(model(x_t)).numpy()[0]

    # 6. diagnóstico clínico
    salida = {}

    for i, enf in enumerate(ENFERMEDADES_71):
        prob = float(pred[i])

        if prob > 0.001:   # 👈 filtro aquí
            thr = UMBRAL_71.get(enf, 0.5)

            salida[enf] = {
                "prob": prob,
                "detectada": prob >= thr,
                "umbral": thr
            }

    # opcional: ordenar por probabilidad
    salida = dict(sorted(salida.items(), key=lambda x: -x[1]["prob"]))

    return salida

In [698]:
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([

    # ── CABECERA ──────────────────────────────────────────────────────────────
    dbc.Row([
        dbc.Col([
            html.H2("🫀 Dashboard de Diagnóstico ECG",
                    style={'color': '#2C5F8A', 'marginBottom': '0'}),
            html.P("Análisis automático de electrocardiogramas mediante Machine Learning",
                   style={'color': '#666', 'marginTop': '4px'})
        ])
    ], style={'padding': '20px 0 10px 0', 'borderBottom': '2px solid #e0e0e0'}),

    html.Br(),

    # ── PANEL DE ENTRADA ──────────────────────────────────────────────────────
    dbc.Card([
        dbc.CardBody([
            html.H5("Datos del paciente", className='card-title'),
            dbc.Row([
                dbc.Col([
                    dcc.Input(
                        id='input-ruta',
                        placeholder="Copia la ruta del fichero .hea de tu ECG",
                        style={
                            'width': '100%',
                            'height': '80px',
                            'lineHeight': '80px',
                            'borderWidth': '2px',
                            'borderStyle': 'dashed',
                            'borderRadius': '10px',
                            'textAlign': 'center',
                            'marginBottom': '10px'
                        },
                        multiple=False
                    )
                ], width=12)
                
            ]),
            html.Br(),
            dbc.Button("🔍 Analizar ECG", id='btn-analizar', color='primary', size='lg'),
        ])
    ], style={'marginBottom': '20px'}),

    # ── MENSAJES DE ERROR ─────────────────────────────────────────────────────
    html.Div(id='div-error'),

    # ── RESUMEN CLÍNICO ───────────────────────────────────────────────────────
    dbc.Card([
        dbc.CardBody([
            html.H5("📊 Resumen clínico", className='card-title'),
            html.Div(id='div-stats-contenido')
        ])
    ]),

    # ── ECG COMPLETO ──────────────────────────────────────────────────────────
    dbc.Card([
        dbc.CardBody([
            html.H5("ECG completo (12 derivaciones)", className='card-title'),
            dcc.Graph(id='grafico-ecg', figure=go.Figure(
                layout=dict(height=900, title='ECG — datos pendientes',
                            plot_bgcolor='white', paper_bgcolor='white')
            ))
        ])
    ], style={'marginBottom': '20px'}),

    # ── LATIDOS SEGMENTADOS ───────────────────────────────────────────────────
    dbc.Card([
        dbc.CardBody([
            html.H5("💓 Latidos segmentados", className='card-title'),
            dcc.Graph(id='grafico-latidos', figure=go.Figure(
                layout=dict(height=600, title='Latidos — datos pendientes',
                            plot_bgcolor='white', paper_bgcolor='white')
            ))
        ])
    ], style={'marginBottom': '20px'}),

    # ── DIAGNÓSTICO ───────────────────────────────────────────────────────────
    dbc.Card([
        dbc.CardBody([

            html.H5("🏥 Diagnóstico", className='card-title'),
            html.Hr(),

            dcc.Graph(
                id='grafico-diagnostico',
                figure=go.Figure(
                    layout=dict(
                        height=350,
                        title='Probabilidad de enfermedad',
                        plot_bgcolor='white',
                        paper_bgcolor='white',
                        margin=dict(l=80, r=80, t=60, b=40)
                    )
                )
            ),

            dbc.Alert(
                "⚠️ Este análisis es orientativo y no sustituye el diagnóstico médico profesional.",
                color='warning',
                style={'marginTop': '10px'}
            )

        ])
    ], style={'marginBottom': '40px'}),

    #Descargar informe
    dbc.Button(
        "📄 Descargar informe",
        id="btn-descargar-pdf",
        color="success",
        size="lg"
    ),

    dcc.Download(id="download-pdf"),
    
    # ── FOOTER ────────────────────────────────────────────────────────────────

    html.Hr(),

    html.Div(
        "TFG — Ana Arregui Beltrán | Curso 2025–2026",
        style={
            'textAlign': 'center',
            'color': '#556B2F',
            'fontSize': '12px',
            'marginBottom': '15px'
        }
    ),

], fluid=True)



In [699]:
@app.callback(
    Output("grafico-ecg", "figure"),
    Output("div-stats-contenido", "children"),
    Output("grafico-latidos", "figure"),
    Output("grafico-diagnostico", "figure"),
    Input("btn-analizar", "n_clicks"),
    State("input-ruta", "value")
)
def analizar(n_clicks, value):

    # ─────────────────────────────
    # CASO INICIAL
    # ─────────────────────────────
    if not n_clicks:
        fig_vacia = go.Figure()
        fig_vacia.update_layout(
            title="Carga un ECG para comenzar",
            height=500,
            plot_bgcolor="white",
            paper_bgcolor="white"
        )
        return fig_vacia, "Esperando análisis...", go.Figure(), go.Figure()

    # ─────────────────────────────
    # SIN INPUT
    # ─────────────────────────────
    if value is None:
        fig_vacia = go.Figure()
        fig_vacia.update_layout(
            title="No se ha cargado ningún ECG",
            height=500,
            plot_bgcolor="white",
            paper_bgcolor="white"
        )
        return fig_vacia, "No hay datos de paciente", go.Figure(), go.Figure()

    try:
        BASE = Path(value)

        # ─────────────────────────────
        # ECG
        # ─────────────────────────────
        L, x, sig, spacing, BASE, n_show, lead_names, fs = leer_ecg(BASE)

        fig_ecg = graficar_ecg(
            L, x, sig,
            spacing,
            BASE,
            n_show,
            lead_names,
            fs
        )

        # ─────────────────────────────
        # RESUMEN + LATIDOS
        # ─────────────────────────────
        resumen, n_latidos = preparar_resumen(BASE)


        stats = generar_resumen_dashboard(resumen, n_latidos)

        diagnostico = predecir_desde_ecg(BASE,resumen) 

        enfermedades = list(diagnostico.keys())
        probs = [diagnostico[e]["prob"] for e in enfermedades]

        detectadas = [
            (enf, info["prob"])
            for enf, info in diagnostico.items()
            if info["detectada"]
        ]

        # ─────────────────────────────
        # LATIDOS (FIGURA 12 DERIVACIONES)
        # ─────────────────────────────
        
        derivaciones=['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

        fig_latidos = make_subplots(
            rows=4,
            cols=3,
            shared_xaxes=True,
            shared_yaxes=False,
            vertical_spacing=0.06,
            horizontal_spacing=0.04,
            subplot_titles=[f"Derivación {derivaciones[i]}" for i in range(12)]
        )

        p_signal_t = sig.T

        ref = p_signal_t[0]
        Rloc = detectar_picos_R(ref, wavelet, nivel)

        tiempo = np.linspace(-0.2, 0.4, pre + post)

        n_deriv = min(12, p_signal_t.shape[0])

        for d in range(n_deriv):

            row = d // 3 + 1
            col = d % 3 + 1

            deriv = p_signal_t[d]
            latidos_d = extraer_latidos(Rloc, deriv, pre, post)
            latidos_d = latidos_d[:MAX_LATIDOS]

            for beat in latidos_d:
                fig_latidos.add_trace(
                    go.Scatter(
                        x=tiempo,
                        y=beat,
                        mode="lines",
                        line=dict(width=1),
                        opacity=0.5,
                        showlegend=False
                    ),
                    row=row,
                    col=col
                )

        fig_latidos.update_layout(
            height=1200,
            title="Latidos segmentados por derivación",
            plot_bgcolor="white",
            paper_bgcolor="white"
        )

        fig_diag = go.Figure(
            go.Bar(
                y=enfermedades,
                x=probs,
                orientation='h',
                marker_color=[
                    '#1D9E75' if d else '#C0C0C0'
                    for d in detectadas
                ],
                text=[f"{p:.0%}" for p in probs],
                textposition='outside'
            )
        )

        fig_diag.update_layout(
            height=800,
            title="Predicción del modelo (71 patologías)",
            xaxis=dict(range=[0, 1], tickformat='.0%'),
            plot_bgcolor='white',
            paper_bgcolor='white'
        )


        return fig_ecg, stats, fig_latidos, fig_diag

    except Exception as e:

        error_msg = traceback.format_exc()

        fig_error = go.Figure()
        fig_error.update_layout(
            title="Error leyendo ECG",
            annotations=[dict(text=error_msg, showarrow=False)]
        )

        return fig_error, error_msg, go.Figure(), go.Figure()
    

#callback descargar informe
@app.callback(
    Output("download-pdf", "data"),
    Input("btn-descargar-pdf", "n_clicks"),

    prevent_initial_call=True
)

def descargar_pdf(n_clicks):

    # ==========================================
    # IMPORTS
    # ==========================================
    
    # ==========================================
    # NOMBRE PDF
    # ==========================================
    nombre_pdf = "informe_ecg.pdf"

    # ==========================================
    # CREAR PDF
    # ==========================================
    c = canvas.Canvas(nombre_pdf, pagesize=letter)

    # ==========================================
    # CABECERA
    # ==========================================
    c.setFont("Helvetica-Bold", 18)
    c.drawString(50, 770, "Informe Automático ECG")

    # ==========================================
    # FECHA
    # ==========================================
    fecha = datetime.datetime.now().strftime("%d/%m/%Y %H:%M")

    c.setFont("Helvetica", 10)
    c.drawString(50, 745, f"Fecha: {fecha}")

    # ==========================================
    # RESUMEN CLÍNICO
    # ==========================================
    c.setFont("Helvetica-Bold", 14)
    c.drawString(50, 700, "Resumen clínico")

    c.setFont("Helvetica", 12)

    c.drawString(70, 670, "• Frecuencia cardíaca: 72 bpm")
    c.drawString(70, 645, "• Eje eléctrico: 45°")
    c.drawString(70, 620, "• Media RR: 0.833 s")
    c.drawString(70, 595, "• SDNN (HRV): 0.045 s")
    c.drawString(70, 570, "• Latidos detectados: 11")

    # ==========================================
    # DIAGNÓSTICO
    # ==========================================
    c.setFont("Helvetica-Bold", 14)
    c.drawString(50, 520, "Diagnóstico")

    c.setFont("Helvetica", 12)

    c.drawString(70, 490, "• NORM: 82%")
    c.drawString(70, 465, "• HYP: 60%")
    c.drawString(70, 440, "• STTC: 45%")

    # ==========================================
    # AVISO LEGAL
    # ==========================================
    c.setFont("Helvetica-Oblique", 10)

    c.drawString(
        50,
        380,
        "Este análisis es orientativo y no sustituye el diagnóstico médico profesional."
    )

    # ==========================================
    # FOOTER
    # ==========================================
    c.setFont("Helvetica", 9)

    c.drawString(
        50,
        40,
        "TFG — Ana Arregui Beltrán | Curso 2025–2026"
    )

    # ==========================================
    # GUARDAR PDF
    # ==========================================
    c.save()

    # ==========================================
    # DESCARGAR PDF
    # ==========================================
    return dcc.send_file(nombre_pdf)

In [700]:
if __name__ == "__main__":
    app.run(debug=True, port=8050)

C:\Users\Ana\AppData\Local\Temp\ipykernel_20780\3863841048.py:22: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

c:\Users\Ana\AppData\Local\Programs\Python\Python312\Lib\site-packages\pywt\_multilevel.py:43: UserWarning:

Level value of 8 is too high: all coefficients will experience boundary effects.



Shape input: (1, 459)
Esperado: 459
